# GH Archive EDA
2025-01-16 一天的資料（24 小時）

In [ ]:
import subprocess
import os

DATA_DIR = '../data/source/gharchive'
DATE = '2025-01-16'

os.makedirs(DATA_DIR, exist_ok=True)

for hour in range(24):
    fname = f'{DATE}-{hour}.json.gz'
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        url = f'https://data.gharchive.org/{fname}'
        print(f'Downloading {fname}...')
        subprocess.run(['curl', '-fL', '--retry', '3', '-o', fpath, url], check=True)
    else:
        print(f'Already exists: {fname}')

print('Done.')

In [ ]:
import pandas as pd
import glob

# 只讀 3 個小時做 EDA，夠看結構
files = sorted(glob.glob(f'{DATA_DIR}/{DATE}-*.json.gz'))[:3]
print(f'Files loaded: {len(files)} (of {len(sorted(glob.glob(f"{DATA_DIR}/{DATE}-*.json.gz")))})')

df = pd.concat([pd.read_json(f, lines=True, compression='gzip') for f in files], ignore_index=True)
print(f'Total records: {len(df):,}')
df.head(3)

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)

In [ ]:
# Null counts
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
pd.DataFrame({'null_count': null_counts, 'null_%': null_pct})

In [ ]:
import matplotlib.pyplot as plt

# Event type distribution
event_counts = df['type'].value_counts()
print(event_counts)

event_counts.plot(kind='bar', figsize=(10, 4), title='Event Type Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# 只看五個核心 event type
CORE_EVENTS = ['WatchEvent', 'ForkEvent', 'PushEvent', 'PullRequestEvent', 'IssuesEvent']
df_core = df[df['type'].isin(CORE_EVENTS)].copy()
print(f'Core events: {len(df_core):,} / {len(df):,} ({len(df_core)/len(df)*100:.1f}%)')
df_core['type'].value_counts()

In [ ]:
# 時間分佈（每小時事件數）
df_core['created_at'] = pd.to_datetime(df_core['created_at'])
df_core['hour'] = df_core['created_at'].dt.hour

hourly = df_core.groupby('hour').size()
hourly.plot(kind='bar', figsize=(12, 4), title='Events per Hour (UTC)')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 最活躍 repo
repo_names = df_core['repo'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
top_repos = repo_names.value_counts().head(20)
print(top_repos)

top_repos.plot(kind='barh', figsize=(10, 6), title='Top 20 Active Repos')
plt.tight_layout()
plt.show()

In [ ]:
# Distinct repos & actors
repos = repo_names.nunique()
actors = df_core['actor'].apply(lambda x: x.get('login') if isinstance(x, dict) else None).nunique()
print(f'Distinct repos:  {repos:,}')
print(f'Distinct actors: {actors:,}')

# PyPI 月下載量分析（2025 全年）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# 載入 PyPI 資料
PYPI_FILE = '../data/source/pypi_monthly_downloads.jsonl'
rows = [json.loads(l) for l in open(PYPI_FILE)]
pypi = pd.DataFrame(rows)
pypi['month'] = pd.to_datetime(pypi['month'])
pypi['downloads'] = pypi['downloads'].astype(int)

print(f'Libraries: {pypi["project"].nunique()}')
print(f'Months: {pypi["month"].nunique()}')
pypi.groupby('project')['downloads'].sum().sort_values(ascending=False)

In [ ]:
# 全年下載量長條圖
total = pypi.groupby('project')['downloads'].sum().sort_values(ascending=True)
total.plot(kind='barh', figsize=(10, 6), title='PyPI Total Downloads 2025')
plt.xlabel('Downloads')
plt.tight_layout()
plt.show()

In [ ]:
# 月下載量成長曲線（top 8 library）
top8 = pypi.groupby('project')['downloads'].sum().nlargest(8).index.tolist()
pivot = pypi[pypi['project'].isin(top8)].pivot(index='month', columns='project', values='downloads')

pivot.plot(figsize=(13, 5), title='Monthly PyPI Downloads 2025 (Top 8)')
plt.ylabel('Downloads')
plt.xlabel('')
plt.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 成長率：2025-01 vs 2025-12
first = pypi[pypi['month'] == '2025-01-01'].set_index('project')['downloads']
last  = pypi[pypi['month'] == '2025-12-01'].set_index('project')['downloads']
growth = ((last - first) / first * 100).sort_values(ascending=False).round(1)
growth.name = 'growth_%'
print(growth.to_string())